# notation_example.py

Generated from `oold-python/examples/notation_example.py`.


Smoke test for the reviewed link declaration notations.

Runnable end-to-end and self-checking, so it doubles as a copy-paste starting
point. Covers the notations discussed in oold-python#107:

1. ``OoldField()`` with no arguments - the link target is inferred from the
   annotation, so the schema IRI is not repeated in Python.
2. ``Link[T]`` / ``LinkList[T]`` as the whole annotation. A link has two types -
   you read a resolved object, you may write that object *or* a reference to it -
   and these carry both, so a type checker accepts an IRI on construction and
   still narrows the read to the target type.
3. **Union arms** mixing a literal, an inline object and a reference.

Run it:

    python examples/notation_example.py

Generated code keeps the unchanged declaration syntax of
``oold.model._descriptor``; ``oold.model._notation`` adds the notations above on
top of it. Which notation supports what is tabulated in
``docs/design/graph-object-binding.md`` section 3.3.


## Environment


In [ ]:
import os

# Selected at oold.model import time, so it has to be set before anything
# imports it. The examples set it too; this covers the ones that do not.
os.environ.setdefault("OOLD_DESCRIPTOR_BINDING", "1")

import piplite

# typing-extensions first: pyodide pins an older one and micropip refuses to
# downgrade a dependency that is already imported.
await piplite.install("typing-extensions>=4.14.0")
await piplite.install("oold", keep_going=True)

# pyodide_http.patch_all() covers requests and urllib, never httpx.
await piplite.install("pyodide-http")
import pyodide_http

pyodide_http.patch_all()
try:
    import httpx
    import requests

    def _get(url, headers=None, verify=None, follow_redirects=True, params=None, **kw):
        return requests.get(url, headers=headers, params=params, allow_redirects=follow_redirects)

    httpx.get = _get
except ImportError:
    pass

import oold

print("oold", oold.__version__)

## Example


In [ ]:
from oold.backend.document_store import SimpleDictDocumentStore
from oold.backend.interface import SetResolverParam, set_resolver
from oold.model import Link, LinkList, OoldField
from oold.model._notation import OoldModel


class Organization(OoldModel):
    id: str
    name: str | None = None
    type: str | None = "ex:Organization"


class Location(OoldModel):
    id: str | None = None  # optional: an inline value may be a blank node
    address: str | None = None
    type: str | None = "ex:Location"


class Person(OoldModel):
    id: str
    name: str | None = None
    type: str | None = "ex:Person"

    # 1. Link[T] / LinkList[T] as the whole annotation - the recommended form.
    #    Reads give the resolved object, writes accept an object, an IRI or a
    #    JSON object, and a type checker sees both (see the coverage note below).
    knows: LinkList["Person"] = OoldField()
    employer: Link[Organization] = OoldField()

    # 2. the plain form: identical at runtime, no range= needed either, but a
    #    type checker only sees list[Person] and so rejects a list of IRIs
    friends: list["Person"] = OoldField()

    # 3. union: literal text | inline object | reference
    location: str | Location | None = OoldField(link=True)


Person.model_rebuild()


def setup_backend() -> SimpleDictDocumentStore:
    store = SimpleDictDocumentStore()
    store.store_json_dicts({
        "ex:bob": {"id": "ex:bob", "name": "Bob", "type": "ex:Person"},
        "ex:carol": {"id": "ex:carol", "name": "Carol", "type": "ex:Person"},
        "ex:acme": {"id": "ex:acme", "name": "ACME", "type": "ex:Organization"},
        "ex:eiffel": {
            "id": "ex:eiffel",
            "address": "Champ de Mars",
            "type": "ex:Location",
        },
    })
    set_resolver(SetResolverParam(iri="ex", resolver=store))
    return store


def main() -> None:
    setup_backend()

    alice = Person(
        id="ex:alice",
        name="Alice",
        knows=["ex:bob", "ex:carol"],  # by IRI, resolved on demand
        employer="ex:acme",
        friends=[Person(id="ex:bob", name="Bob")],  # or by object
    )

    print("1. Link[T] / LinkList[T] - target inferred, and statically typed")
    assert alice.knows[0].name == "Bob"
    assert isinstance(alice.knows[0], Person)  # a real Person, not a proxy
    print("   knows[0].name          =", alice.knows[0].name)
    print("   isinstance(.., Person) =", isinstance(alice.knows[0], Person))

    print("\n2. the plain list[T] form - identical at runtime")
    assert isinstance(alice.employer, Organization)
    assert alice.employer.name == "ACME"
    assert isinstance(alice.friends[0], Person)
    print("   employer.name          =", alice.employer.name)
    print("   friends[0].name        =", alice.friends[0].name)

    print("\n3. union arms: text | inline object | reference")
    text = Person(id="ex:p-text", location="at the Eiffel Tower")
    ref = Person(id="ex:p-ref", location={"@id": "ex:eiffel"})
    inline = Person(
        id="ex:p-inline",
        location={"id": "ex:office", "address": "Main St 1", "type": "ex:Location"},
    )
    blank = Person(id="ex:p-blank", location={"address": "no id", "type": "ex:Location"})

    # A union field is str | Location | None, so narrow it to a local before
    # dereferencing - the same hygiene any union needs, and what lets a type
    # checker follow along.
    ref_loc, inline_loc, blank_loc = ref.location, inline.location, blank.location
    assert text.location == "at the Eiffel Tower"  # stays a literal
    assert isinstance(ref_loc, Location)  # resolved reference
    assert isinstance(inline_loc, Location) and isinstance(blank_loc, Location)
    assert ref_loc.address == "Champ de Mars"
    assert inline_loc.address == "Main St 1"
    assert blank.link_iris("location") is None  # no IRI -> blank node
    print("   text   ->", repr(text.location))
    print("   ref    ->", ref_loc.address)
    print("   inline ->", inline_loc.address)
    print("   blank  ->", blank_loc.address, "(no IRI)")

    print("\n4. serialisation: links to IRIs; references boxed where a literal arm exists")
    dumped = alice.model_dump(exclude_none=True)
    assert dumped["knows"] == ["ex:bob", "ex:carol"]
    assert dumped["employer"] == "ex:acme"
    # boxed as {"@id": ...} because the field also accepts a literal
    assert ref.model_dump(exclude_none=True)["location"] == {"@id": "ex:eiffel"}
    assert isinstance(blank.model_dump(exclude_none=True)["location"], dict)
    print("   alice ->", dumped)
    print("   ref   ->", ref.model_dump(exclude_none=True))
    print("   blank ->", blank.model_dump(exclude_none=True))

    print("\n5. lazy resolution and query DSL")
    lazy = Person(id="ex:lazy", knows=["ex:bob"])
    assert lazy.link_iris("knows") == ["ex:bob"]  # inspect without resolving
    # The class-level DSL builds a Condition at runtime, but a type checker
    # sees BaseModel.__eq__ and reads this as bool. The subscript overloads
    # accept bool for that reason, so Person[cond] keeps its result type.
    condition = Person.name == "Bob"
    assert condition.field == "name"
    print("   link_iris('knows')     =", lazy.link_iris("knows"))
    print("   Person.name == 'Bob'   =", condition)

    print("\n6. de-serialisation: every union arm survives a round trip")
    for label, value, expected in [
        ("text     ", "at the Eiffel Tower", str),
        ("reference", {"@id": "ex:eiffel"}, Location),
        ("inline   ", {"address": "Main St 1", "type": "ex:Location"}, Location),
    ]:
        original = Person(id="ex:rt", location=value)
        payload = original.model_dump(exclude_none=True)
        restored = Person(**payload)
        assert isinstance(restored.location, expected), label
        shown = str(payload["location"])[:34]
        print(f"   {label} {shown:36} -> {type(restored.location).__name__}")

    restored = Person(**alice.model_dump(exclude_none=True))
    assert [x.id for x in restored.knows] == ["ex:bob", "ex:carol"]
    restored_employer = restored.employer  # to-one link: narrow before use
    assert restored_employer is not None and restored_employer.name == "ACME"
    print("   lists and to-one links round trip too")

    print("\nALL CHECKS PASSED")

In [ ]:
main()